In [ ]:
#Validation

# Cell 12: Run model.val() for official mAP metrics
val_model = YOLO(str(best_checkpoint_path))
metrics = val_model.val(
    data=str(data_yaml_path),
    imgsz=img_size,
    conf=conf_default,
    iou=nms_iou,
    device=compute_device
)

map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)

print("\n" + "=" * 50)
print("  YOLO-Only Validation Results")
print("=" * 50)
print(f"  mAP@50       : {map50:.4f}")
print(f"  mAP@50-95    : {map50_95:.4f}")
print(f"  Precision    : {precision:.4f}")
print(f"  Recall       : {recall:.4f}")
print("=" * 50)

# Save for Member 3
with open(detection_results_dir / "yolo_baseline_metrics.json", "w") as f:
    json.dump({
        "model": yolo_model,
        "image_size": img_size,
        "confidence_threshold": conf_default,
        "iou_threshold_nms": nms_iou,
        "mAP_at_0.50": map50,
        "mAP_at_0.50_to_0.95": map50_95,
        "precision": precision,
        "recall": recall,
    }, f, indent=2)

#Qualitative Samples

# Cell 13: 6-image grid — predictions vs ground truth
sample_indices = random.sample(range(len(val_images)), 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("YOLOv11s Predictions on Fused IR+Visible Input\n"
             "(Red = prediction with confidence | Green dashed = ground truth)",
             fontsize=12, fontweight="bold")
axes_flat = axes.flatten()

q_model = YOLO(str(best_checkpoint_path))

for pos, idx in enumerate(sample_indices):
    img_path = val_images[idx]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

    pred = q_model.predict(source=str(img_path), conf=conf_default, iou=nms_iou,
                           imgsz=img_size, verbose=False)
    boxes = pred[0].boxes.xyxy.cpu().numpy()
    confs = pred[0].boxes.conf.cpu().numpy()
    gt = load_gt(val_labels_dir / (img_path.stem + ".txt"), img_size)

    axes_flat[pos].imshow(img)
    for g in gt:
        rect = patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                                 linewidth=2, edgecolor="lime", linestyle="--", facecolor="none")
        axes_flat[pos].add_patch(rect)
    for b, c in zip(boxes, confs):
        rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                 linewidth=2, edgecolor="red", facecolor="none")
        axes_flat[pos].add_patch(rect)
        axes_flat[pos].text(b[0], b[1]-5, f"{c:.2f}", color="red",
                            fontsize=8, fontweight="bold")
    axes_flat[pos].set_title(f"{img_path.name} — {len(boxes)} det / {len(gt)} GT", fontsize=10)
    axes_flat[pos].axis("off")

plt.tight_layout()
plt.savefig(detection_results_dir / "qualitative_samples.png", dpi=150, bbox_inches="tight")
plt.show()

Bridge Export to SAM 2
Save bbox detections in xyxy absolute pixels
Output keyed by image filename, ready for SAM2ImagePredictor.predict(box=...).


# Cell 14: Export yolo_detections.json for Member 1's SAM 2 stage
export_model = YOLO(str(best_checkpoint_path))
export_dict = {
    "metadata": {
        "model": yolo_model,
        "checkpoint_path": str(best_checkpoint_path),
        "image_size": img_size,
        "confidence_threshold": conf_default,
        "iou_threshold_nms": nms_iou,
        "max_detections": max_det,
        "format": "xyxy_absolute_pixels",
        "num_images": len(val_images),
    },
    "detections": {}
}

total = 0
zero_count = 0
for i, img_path in enumerate(val_images):
    out = export_model.predict(source=str(img_path), conf=conf_default, iou=nms_iou,
                               imgsz=img_size, max_det=max_det, verbose=False)
    boxes = out[0].boxes.xyxy.cpu().numpy()
    confs = out[0].boxes.conf.cpu().numpy()

    export_dict["detections"][img_path.name] = {
        "image_size_hw": [img_size, img_size],
        "boxes_xyxy": boxes.round(2).tolist(),
        "confidences": confs.round(4).tolist(),
    }
    total += len(boxes)
    if len(boxes) == 0: zero_count += 1
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(val_images)} | {total} detections so far")

bridge_path = detection_results_dir / "yolo_detections.json"
with open(bridge_path, "w") as f:
    json.dump(export_dict, f, indent=2)

print("\n" + "=" * 60)
print("  Bridge Export Complete")
print("=" * 60)
print(f"  Images          : {len(val_images)}")
print(f"  Detections      : {total}")
print(f"  Zero-detection  : {zero_count}")
print(f"  Avg det/img     : {total/len(val_images):.2f}")
print(f"  Output          : {bridge_path}")
print(f"  File size       : {bridge_path.stat().st_size/1e6:.2f} MB")

Loading snippet for Member 1's SAM 2 stage
import json, numpy as np
with open("yolo_detections.json") as f:
    bridge = json.load(f)

# For one image
rec = bridge["detections"]["010001.jpg"]
boxes = np.array(rec["boxes_xyxy"])

sam2_predictor.set_image(fused_rgb)
masks, iou, _ = sam2_predictor.predict(
    point_coords=None, point_labels=None,
    box=boxes, multimask_output=False
)
#Summary

# Cell 15: Print final summary and saved artefacts
print("\n" + "#" * 60)
print("  MEMBER 2 — YOLO DETECTION COMPLETE")
print("#" * 60)
print(f"\n  Model          : {yolo_model}")
print(f"  Image size     : {img_size}")
print(f"  Confidence     : {conf_default}")
print(f"  NMS IoU        : {nms_iou}")
print(f"\n  mAP@50         : {map50:.4f}")
print(f"  mAP@50-95      : {map50_95:.4f}")
print(f"  Precision      : {precision:.4f}")
print(f"  Recall         : {recall:.4f}")

print("\n  Saved artefacts:")
for p in sorted(detection_results_dir.rglob("*")):
    if p.is_file() and p.suffix in (".csv", ".json", ".png"):
        print(f"    {p.relative_to(detection_results_dir)}  ({p.stat().st_size/1e6:.2f} MB)")